In [ ]:
import sys
import csv
import os
import json

datadir = '/Users/dragonjohn/Documents/DevEnv/Python/MLCR2malwareDetect/dataset/'
train_CSV_file = 'train.csv'
output_train_csv_file = 'train_data.csv'
output_test_csv_file = 'test_data.csv'

info_feature_selected = [1, 4, 7, 8, 9, 11, 12, 14, 18, 23, 26, 29, 30, 32, 33, 34]

if len(sys.argv)>=2:
    datadir = sys.argv[1]
else:
    print "no dataset directory provided"
    exit(1)

def select_info_feature():
    selected_label = []
    with open('info_label.txt', 'rb') as info_label_file:
        labels = info_label_file.readlines()
        for idx, row in enumerate(labels):
            if idx in info_feature_selected:
                selected_label.append(row.rstrip())
    return selected_label

def get_info_train_dataset(selected_features):
    train_x = []
    title_row = ['label']+ selected_features
    train_x.append(title_row)
    with open(datadir + train_CSV_file, 'rb') as csvfile:
        spamreader = csv.reader(csvfile, delimiter = ',', quotechar='"')
        for row in spamreader:
            data_row = []
            data_row.append(row[1])
            with open(datadir + 'train/' + row[0] + '/info') as info_file:
                data = json.load(info_file)
                for key, value in data.items():
                    if key in selected_features:
                        data_row.append(value)
            train_x.append(data_row)
    return train_x

def get_info_test_dataset(selected_features):
    test_x = []
    for filefolder in os.listdir(datadir + 'test/'):
        if filefolder.startswith('.'):
            continue
        data_row = []
        data_row.append(filefolder)
        with open(datadir + 'test/' + filefolder +'/info') as info_file:
            data = json.load(info_file)
            for key, value in data.items():
                if key in selected_features:
                    data_row.append(value)
        test_x.append(data_row)
    return test_x

In [ ]:
import pandas
f1 = pandas.read_csv('dataset/train_info.csv', comment="#", header=None, engine='c', dtype={0: 'object'})
#f1.columns
#f1.columns = ["label"]+f1.columns.tolist()
md5 = f1.ix[1:,:0]
md5

#column_attr = f1.ix[0:0,:]
#column_attr[0][0]
column_attr = f1.ix[0:0,:]
if 'md5' == column_attr[0][0]:
    print "True"

In [ ]:
f1 = f1.ix[1:,0:]
#f1
label = f1.ix[0:,1:1]
#label
import numpy as np
data = f1.ix[0:,2:]
da = np.array(data,dtype="float32")
#da

In [ ]:
import numpy as np
import pandas

def parsefile(filepath):
    f = pandas.read_csv(filepath, comment="#", header=None, engine='c', dtype={0: 'object'})

    md5 = None
    label = None
    data = None

    drop_cols = [54,66,68,69,72,74,81,83,85,87,89,90,95,98,106,109,111,116,117,135,137,
                 140,142,147,149,160,163,165,166,168,171,172,175,176,180,181,185,187,194,205,209]
    
    #drop_cols2 = [62] # 19,42,46,62,63,101,102,163,164,166
    
    column_attr = f.ix[0:0,:]
    if 'md5' == column_attr[0][0]: #train dataset with title and label
        f = f.ix[1:,0:] #remove first row
        la = f.ix[0:,1:1]
        label = np.array(la,dtype="int")
        da = f.ix[0:,2:].replace(np.nan,0, regex=True)
    else:
        da = f.ix[0:,1:].replace(np.nan,0, regex=True)
    md5 = f.ix[0:,:0]
    #drop 
    da.drop(da.columns[drop_cols],axis=1,inplace=True)
    #da.drop(da.columns[drop_cols2],axis=1,inplace=True)
    data = np.array(da)
    return data, label, md5

In [ ]:
#LOAD TRAIN DATA
data, label, md5 = parsefile('dataset/train_info+import+asm+rom_select5.csv')
#LOAD TEST DATA
tdata, tlable, tmd5 = parsefile('dataset/test_new_data_info+import+asm+rom_select4.csv')

# data : ndarray
# label : Dataframe
##import numpy as np
##np.asarray(label.as_matrix().transpose().tolist()[0])
label

In [ ]:
data

In [ ]:
import matplotlib.pyplot as plt

import numpy as np

from sklearn.datasets import make_blobs
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss
#X = data
#y = label
X, y = make_blobs(n_samples=10, n_features=2, random_state=42,
                  cluster_std=5.0)
X_train, y_train = X[:600], y[:600]
X_valid, y_valid = X[600:800], y[600:800]
X_train_valid, y_train_valid = X[:800], y[:800]
X_test, y_test = X[800:], y[800:]
y

In [ ]:
from sklearn.cross_validation import cross_val_score
from sklearn.datasets import make_blobs
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier

X, y = make_blobs(n_samples=10000, n_features=10, centers=100,
    random_state=0)
                        
#clf = RandomForestClassifier(n_estimators=10, max_depth=None,
#    min_samples_split=1, random_state=0)
#scores = cross_val_score(clf, X, y)
#scores.mean()                             


clf = ExtraTreesClassifier(n_estimators=10, max_depth=None,
    min_samples_split=1, random_state=0)
scores = cross_val_score(clf, X, y)
scores.mean() > 0.999

y.shape

In [ ]:
X.shape

In [ ]:
X

In [ ]:
y

In [ ]:
import numpy as np
from sklearn.cross_validation import cross_val_score
from sklearn.datasets import make_blobs
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier

X = data
y = label.transpose()[0]
y

In [ ]:
X

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
clf = ExtraTreesClassifier(n_estimators=250, max_depth=None, 
    criterion='entropy',
    min_samples_split=2, random_state=0)
scores = cross_val_score(clf, X, y,cv=10, n_jobs=-1)
scores.mean()

In [ ]:
print(__doc__)

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.ensemble import ExtraTreesClassifier

clf.fit(X, y)
importances = clf.feature_importances_
std = np.std([tree.feature_importances_ for tree in clf.estimators_],
             axis=0)
indices = np.argsort(importances)[::-1]

# Print the feature ranking
print("Feature ranking:")

for f in range(X.shape[1]):
    print("%d. feature %d (%f)" % (f + 1, indices[f], importances[indices[f]]))

# Plot the feature importances of the forest
plt.figure()
plt.title("Feature importances")
plt.bar(range(X.shape[1]), importances[indices],
       color="r", yerr=std[indices], align="center")
plt.xticks(range(X.shape[1]), indices)
plt.xlim([-1, X.shape[1]])
plt.show()

In [ ]:
#clf = RandomForestClassifier(n_estimators=10, max_depth=None,
#    min_samples_split=3, random_state=0)
#scores = cross_val_score(clf, X, y)
#scores.mean()    

In [ ]:
#from sklearn.ensemble import GradientBoostingClassifier
#clf = GradientBoostingClassifier(n_estimators=250, max_depth=None,
#    min_samples_split=3, random_state=0)
#scores = cross_val_score(clf, X, y)
#scores.mean()   

In [ ]:
clf = ExtraTreesClassifier(n_estimators=2500, max_depth=None, 
    criterion='entropy',
    min_samples_split=2, random_state=0)
scores = cross_val_score(clf, X, y, cv=100, scoring='roc_auc', n_jobs=-1)
scores.mean()

In [ ]:
scores

In [ ]:
tdata

In [ ]:
tdata.shape

In [ ]:
#from sklearn.ensemble import RandomForestRegressor
clf.fit(data, label)
clf.predict(tdata)

In [ ]:
import numpy as np
from sklearn import metrics
y = np.array([1, 1, 2, 2])
pred = np.array([0.1, 0.4, 0.35, 0.8])
fpr, tpr, thresholds = metrics.roc_curve(y, pred, pos_label=2)
metrics.auc(fpr, tpr)

In [ ]:
from numpy import genfromtxt, savetxt
savetxt('dataset/submission20160717_1.csv', clf.predict(tdata), delimiter=',', fmt='%f')

In [ ]:
import os

# /Users/dragonjohn/Documents/DevEnv/Python/MLCR2malwareDetect/RyuMalwareDetect/
asmopcode = 'asm_opcode.txt'

opcode_list = []

with open(asmopcode, 'rb') as content:
    for line in content.readlines():
        opcode = line.split(' - ')
        if '/' in opcode[0]:
            opcode_list.extend([opcode[0].split('/')[0], opcode[0].split('/')[1]])
        else:
            opcode_list.append(opcode[0])
            
print opcode_list



In [ ]:
import os

asmfilepath = '../dataset-asm-reaserch/train/'
feature_list = []
feature_list = [["md5"] + opcode_list]

opcode_list = ['\t\t' + op.lower() for op in opcode_list]

print opcode_list

In [ ]:
asmopcode = 'asm_opcode3.txt'

with open(asmopcode, 'rb') as content:
    for line in content.read().splitlines():
        opcode_list.append(line.lower())
        feature_list[0]+=[line]
            
print feature_list

In [ ]:
print opcode_list

In [ ]:
for asmfile in os.listdir(asmfilepath):
    # print asmfile
    # The following table provides a list of x86-Assembler mnemonics, that is not complete. 
    # Most of them can be found, for others see at www.intel.com
    # http://www.mathemainzel.info/files/x86asmref.html
    # count add, sub, mov, xor, push, pop, cmp, jz, jnz, je, jne, ja, 
    with open(asmfilepath+asmfile) as filecontent:
        content_feature = [0] * len(opcode_list)

        for line in filecontent:
            for idx, op in enumerate(opcode_list):
                if op in line.lower():
                    content_feature[idx] +=1
        feature_list += ([[asmfile.split('.asm')[0]] + content_feature])
    
print feature_list


In [ ]:
import csv
output_train_asm = 'dataset/train_asm.csv'
with open(output_train_asm, 'wb') as output:
    writer = csv.writer( output, delimiter=",")
    for row in feature_list:
        writer.writerow(row)